# Лабораторная работа № 2
## Колоночное хранение, партиционирование и бенчмаркинг Parquet

**Дисциплина:** «Методы обработки больших данных»
**Направление:** 44.03.05 Педагогическое образование
**Профиль:** «Информатика и дополнительное образование (робототехника)»
**Этап конвейера:** `Collect → Store → Process`
**Среда:** Google Colab · PySpark 4.2.0 · `local[*]` · Parquet · Snappy / ZSTD / GZIP

---

### Цель работы

Освоить колоночное хранение телеметрии в формате Parquet, физическое партиционирование набора
и экспериментальное сравнение объёма и времени чтения CSV и Parquet. Понять, за счёт чего
возникает выигрыш и когда он исчезает.

### Что нужно сдать

1. Этот ноутбук с выполненными заданиями (`.ipynb`, все ячейки выполнены сверху вниз).
2. Ссылку на ноутбук в GitHub или Colab.
3. Заполненную форму **TEACH CARD** в конце ноутбука.

### Критерии оценки — ровно 10 баллов

| Часть | Содержание | Баллы |
|---|---|---|
| 1 | Три физические версии набора: CSV, Parquet, partitioned Parquet | 2 |
| 2 | Размеры, коэффициент сжатия, влияние энтропии и выбора кодека | 2 |
| 3 | Воспроизводимый бенчмарк чтения: ≥ 3 повтора, медиана, честное сравнение | 2 |
| 4 | Column pruning, partition pruning, проблема мелких файлов | 2 |
| 5 | Заполненная форма TEACH CARD | 2 |

> **Правила выполнения.** Не ищите готовые решения — пользуйтесь `help()` и документацией Spark.
> Ячейки с `assert` — самопроверка: если проверка прошла, задание засчитано. Числовые значения
> времени зависят от runtime Colab, поэтому оценивается методика измерения, а не абсолютные цифры.

### Источники

* Balapriya C. `data-science-tutorials` — <https://github.com/balapriyac/data-science-tutorials>
  (`pyspark/pyspark_write_parquet.ipynb`)
* Piotr Szul. `spark-tutorial` — <https://github.com/piotrszul/spark-tutorial>
* Apache Parquet — <https://parquet.apache.org/docs/>
* Spark. Parquet Files — <https://spark.apache.org/docs/latest/sql-data-sources-parquet.html>

---
## Задание 0. Подготовка среды и данных

### Минимальные теоретические сведения

**Колоночное хранение.** Parquet размещает рядом значения одной колонки, а не одной записи.
Отсюда два следствия: однородные данные сжимаются в разы лучше, а запрос, которому нужны две
колонки из десяти, читает только эти две — это **column pruning**.

**Партиционирование.** `partitionBy("robot_id")` раскладывает данные по каталогам вида
`robot_id=robot-7/`. Запрос с фильтром по этой колонке пропускает лишние каталоги целиком —
это **partition pruning**. Значение ключа при этом физически не хранится в файлах: оно
закодировано в имени каталога.

**Метрики работы:**

коэффициент уменьшения объёма  $C = S_{CSV} / S_{Parquet}$

ускорение чтения  $Speedup = T_{CSV} / T_{Parquet}$

Результаты зависят от runtime Colab и файлового кэша, поэтому каждое измерение повторяется
не менее трёх раз, а итогом берётся **медиана** (она устойчивее среднего к единичному выбросу
при первом «холодном» чтении).

In [ ]:
!pip -q install "pyspark==4.2.0" pandas pyarrow

In [ ]:
import shutil
import time
import statistics
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab02_ParquetBenchmark")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:", spark.version)

ROOT = Path("/content/lab02_storage")
if not Path("/content").exists():          # запуск вне Colab
    ROOT = Path("./lab02_storage")

CSV_DIR         = ROOT / "csv"
PARQUET_DIR     = ROOT / "parquet"
PARTITIONED_DIR = ROOT / "parquet_partitioned"

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
print("Каталог хранения:", ROOT)

### Генерация телеметрии

Набор порождается детерминированно, внешний датасет не нужен. Обратите внимание на колонку
`payload`: это SHA-256 от идентификатора, то есть **высокоэнтропийная** строка без повторов.
Она добавлена намеренно — в части 2 мы посмотрим, что она делает со сжатием.

DataFrame кэшируется: иначе Spark пересчитает его заново при каждой из трёх записей.

In [ ]:
N = 300_000

df = (
    spark.range(N)
    .withColumn("robot_id",    F.concat(F.lit("robot-"), (F.col("id") % 12).cast("string")))
    .withColumn("sensor",      F.when(F.col("id") % 2 == 0, F.lit("imu")).otherwise(F.lit("motor")))
    .withColumn("ts_ms",       (F.lit(1_720_000_000_000) + F.col("id") * 100).cast("long"))
    .withColumn("temperature", F.lit(40.0) + (F.col("id") % 150) * 0.08)
    .withColumn("vibration",   F.abs(F.sin(F.col("id") / 15.0)) + (F.col("id") % 11) * 0.002)
    .withColumn("payload",     F.sha2(F.col("id").cast("string"), 256))
    .drop("id")
    .cache()
)

print("Строк:", df.count())
df.printSchema()
df.show(3, truncate=False)

---
# Часть 1. Три физические версии набора (2 балла)

Один и тот же DataFrame нужно записать тремя способами и дальше сравнивать их между собой.

## Упражнение 1.1. CSV

Запишите `df` в CSV в каталог `CSV_DIR`. Режим записи — `overwrite`, заголовок включён
(`option("header", True)`).

In [ ]:
# TODO: df.write ... .csv(str(CSV_DIR))

files = sorted(p.name for p in CSV_DIR.iterdir())
print("Файлов в каталоге CSV:", len(files))
print(files[:5])
assert CSV_DIR.exists() and any(p.suffix == ".csv" for p in CSV_DIR.iterdir())

> **Обратите внимание.** Spark пишет не один файл, а каталог: по одному файлу `part-*` на каждую
> partition плюс маркер `_SUCCESS`. Так устроены все распределённые форматы — единый файл нельзя
> писать параллельно с нескольких узлов.

## Упражнение 1.2. Parquet со сжатием Snappy

Запишите тот же `df` в Parquet в каталог `PARQUET_DIR`, явно указав кодек
`option("compression", "snappy")`.

In [ ]:
# TODO: df.write ... .parquet(str(PARQUET_DIR))

parts = [p.name for p in PARQUET_DIR.iterdir() if p.suffix == ".parquet"]
print("Файлов parquet:", len(parts))
print(parts[:3])
assert len(parts) > 0
assert all("snappy" in p for p in parts), "В имени файла должен быть виден кодек"

## Упражнение 1.3. Partitioned Parquet

Запишите набор в `PARTITIONED_DIR` с физическим партиционированием по `robot_id`
(`partitionBy`). Затем выведите список созданных каталогов и проверьте, что их ровно 12 —
по числу роботов.

In [ ]:
# TODO: df.write ... .partitionBy(...).parquet(str(PARTITIONED_DIR))

part_dirs = sorted(p.name for p in PARTITIONED_DIR.iterdir() if p.is_dir())
print("Каталогов:", len(part_dirs))
print(part_dirs[:4])

assert len(part_dirs) == 12
assert all(d.startswith("robot_id=") for d in part_dirs)

## Упражнение 1.4

Прочитайте один файл из каталога `robot_id=robot-7` напрямую (`spark.read.parquet` по полному
пути к каталогу партиции) и посмотрите его схему. Есть ли в ней колонка `robot_id`?
Ответьте письменно, почему.

In [ ]:
one_partition = spark.read.parquet(str(PARTITIONED_DIR / "robot_id=robot-7"))
one_partition.printSchema()

ANSWER_1_4 = """
TODO: ваш ответ (2-4 предложения)
"""
print(ANSWER_1_4)

assert "robot_id" not in one_partition.columns

---
# Часть 2. Объём хранения (2 балла)

## Упражнение 2.1. Размеры и коэффициент сжатия

Напишите функцию `dir_size(path)`, возвращающую суммарный размер всех файлов каталога в байтах
(каталог вложенный, поэтому нужен рекурсивный обход — подсказка: `Path.rglob`).

Выведите таблицу размеров трёх версий и рассчитайте коэффициент
$C = S_{CSV} / S_{Parquet}$.

In [ ]:
def dir_size(path: Path) -> int:
    """Суммарный размер всех файлов внутри каталога, включая вложенные."""
    # TODO
    raise NotImplementedError


sizes = {
    "CSV":                 dir_size(CSV_DIR),
    "Parquet":             dir_size(PARQUET_DIR),
    "Partitioned Parquet": dir_size(PARTITIONED_DIR),
}
for name, size in sizes.items():
    print(f"{name:22s}: {size / 1024 / 1024:7.2f} MiB")

compression_ratio = sizes["CSV"] / sizes["Parquet"]
print(f"\nКоэффициент C = S_CSV / S_Parquet = {compression_ratio:.2f}x")

assert sizes["Parquet"] < sizes["CSV"], "Parquet должен быть компактнее CSV"
assert compression_ratio > 1.0

## Упражнение 2.2. Энтропия и сжимаемость

Коэффициент из предыдущего упражнения получился скромным. Причина — колонка `payload`:
это SHA-256, то есть практически случайная строка. Данные с высокой энтропией не сжимаются
принципиально: в них нет повторов, которые можно было бы закодировать короче.

Запишите **тот же набор без колонки `payload`** в CSV и в Parquet, посчитайте коэффициент
заново и сравните с предыдущим.

In [ ]:
NOPAY_CSV = ROOT / "csv_nopayload"
NOPAY_PQ  = ROOT / "parquet_nopayload"

df_nopay = df.drop("payload")

# TODO: запишите df_nopay в CSV (с заголовком) и в Parquet (snappy)

sizes_nopay = {"CSV": dir_size(NOPAY_CSV), "Parquet": dir_size(NOPAY_PQ)}
ratio_nopay = sizes_nopay["CSV"] / sizes_nopay["Parquet"]

print(f"Без payload: CSV {sizes_nopay['CSV']/1024/1024:.2f} MiB, "
      f"Parquet {sizes_nopay['Parquet']/1024/1024:.2f} MiB")
print(f"Коэффициент без payload: {ratio_nopay:.2f}x  (с payload было {compression_ratio:.2f}x)")

ANSWER_2_2 = """
TODO: объясните, почему коэффициент изменился именно так (3-5 предложений)
"""
print(ANSWER_2_2)

assert ratio_nopay > compression_ratio, "Без высокоэнтропийной колонки выигрыш должен вырасти"

## Упражнение 2.3. Сравнение кодеков

Запишите `df_nopay` в Parquet четырьмя кодеками: `uncompressed`, `snappy`, `gzip`, `zstd`.
Для каждого измерьте время записи и итоговый размер. Заполните словарь `codec_stats`
в формате `{кодек: (секунды, байты)}`.

In [ ]:
codec_stats = {}

for codec in ["uncompressed", "snappy", "gzip", "zstd"]:
    out = ROOT / f"pq_{codec}"
    # TODO: замерьте время записи df_nopay с этим кодеком и размер каталога
    pass

print(f"{'кодек':<14}{'время, с':>10}{'размер, MiB':>14}{'C к uncompressed':>20}")
base = codec_stats["uncompressed"][1]
for codec, (t, s) in codec_stats.items():
    print(f"{codec:<14}{t:>10.2f}{s/1024/1024:>14.2f}{base/s:>20.2f}")

assert set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"}
assert codec_stats["gzip"][1] < codec_stats["snappy"][1], "gzip должен сжимать сильнее snappy"

**Вопрос 2.4.** Почему Snappy используется по умолчанию, хотя gzip сжимает заметно сильнее?
Ответьте письменно.

In [ ]:
ANSWER_2_4 = """
TODO: ваш ответ
"""
print(ANSWER_2_4)

---
# Часть 3. Бенчмарк чтения (2 балла)

## Упражнение 3.1. Функция измерения

Напишите функцию `benchmark_read(fmt, path, predicate=None, repeats=3, schema=None)`, которая:

1. `repeats` раз читает набор из `path` в формате `fmt` (`"csv"` или `"parquet"`);
2. при заданном `predicate` применяет фильтр;
3. выполняет действие `count()` — **без него из-за ленивости ничего не прочитается**;
4. возвращает кортеж `(медиана времени, число строк)`.

Для CSV используйте `option("header", True)`; параметр `schema` пока оставьте
необязательным — он понадобится в упражнении 3.3.

In [ ]:
def benchmark_read(fmt, path, predicate=None, repeats=3, schema=None):
    """Возвращает (медианное время чтения, число строк)."""
    times, counts = [], []
    for _ in range(repeats):
        t0 = time.perf_counter()
        # TODO: прочитайте данные нужного формата, примените predicate, вызовите count()
        raise NotImplementedError
    return statistics.median(times), counts[-1]


t, n = benchmark_read("parquet", PARQUET_DIR, repeats=3)
print(f"Чтение всего Parquet: {t:.3f} с, строк {n}")
assert n == N

## Упражнение 3.2. Сравнение трёх версий

Выполните одинаковый запрос `robot_id == "robot-7"` к каждой из трёх версий набора, минимум
по три повтора. Рассчитайте ускорение относительно CSV.

In [ ]:
predicate = F.col("robot_id") == "robot-7"

csv_t,  csv_n  = benchmark_read("csv",     CSV_DIR,         predicate)
pq_t,   pq_n   = benchmark_read("parquet", PARQUET_DIR,     predicate)
part_t, part_n = benchmark_read("parquet", PARTITIONED_DIR, predicate)

print(f"CSV                 : {csv_t:6.3f} с, строк {csv_n}")
print(f"Parquet             : {pq_t:6.3f} с, строк {pq_n}")
print(f"Partitioned Parquet : {part_t:6.3f} с, строк {part_n}")
print()
print(f"Speedup CSV -> Parquet             : {csv_t / pq_t:.2f}x")
print(f"Speedup CSV -> Partitioned Parquet : {csv_t / part_t:.2f}x")

assert csv_n == pq_n == part_n == N // 12, "Все три версии должны вернуть одинаковое число строк"

## Упражнение 3.3. Честное сравнение

В упражнении 3.1 CSV читался с `inferSchema=True`. Это удобно, но нечестно по отношению к CSV:
чтобы вывести типы, Spark делает **дополнительный полный проход** по файлу. Получается, что
мы сравниваем Parquet с CSV, прочитанным дважды.

Повторите замер CSV с **явно заданной схемой** — возьмите её у уже прочитанного Parquet
(`spark.read.parquet(...).schema`) — и сравните три числа: CSV с выводом типов, CSV со схемой
и Parquet.

In [ ]:
explicit_schema = spark.read.parquet(str(PARQUET_DIR)).schema

# TODO: замерьте чтение CSV с явной схемой (schema=explicit_schema)
csv_schema_t, csv_schema_n = None, None

print(f"CSV + inferSchema : {csv_t:6.3f} с")
print(f"CSV + явная схема : {csv_schema_t:6.3f} с")
print(f"Parquet           : {pq_t:6.3f} с")
print(f"\nЧестный speedup CSV -> Parquet: {csv_schema_t / pq_t:.2f}x")

ANSWER_3_3 = """
TODO: насколько изменился выигрыш Parquet и почему такое уточнение важно?
"""
print(ANSWER_3_3)

assert csv_schema_n == csv_n
assert csv_schema_t < csv_t, "Без inferSchema чтение CSV должно ускориться"

---
# Часть 4. Pruning и мелкие файлы (2 балла)

## Упражнение 4.1. Partition pruning

Выполните `explain(mode="formatted")` для запроса к partitioned Parquet с фильтром по
`robot_id` и найдите в плане строку `PartitionFilters`.

In [ ]:
(
    spark.read.parquet(str(PARTITIONED_DIR))
    .filter(F.col("robot_id") == "robot-7")
    .select("robot_id", "temperature")
    .explain(mode="formatted")
)

In [ ]:
ANSWER_4_1 = """
TODO: что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?
"""
print(ANSWER_4_1)

## Упражнение 4.2. Column pruning в цифрах

Сравните время двух запросов к обычному (непартиционированному) Parquet:

* агрегация **только** по `temperature`;
* агрегация по `temperature` **и** по тяжёлой высокоэнтропийной колонке `payload`.

Второй запрос вынуждает Spark поднять с диска колонку, которая занимает большую часть объёма
файла. Каждый запрос — минимум три повтора, берите медиану.

Дополнительно выведите `explain` обоих вариантов и сравните строки `ReadSchema`.

In [ ]:
def timed(fn, repeats=3):
    """Медианное время выполнения fn()."""
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return statistics.median(ts)


def read_with_payload():
    # TODO: agg по temperature И по payload (например, F.max("payload"))
    raise NotImplementedError


def read_one_column():
    # TODO: agg только по temperature
    raise NotImplementedError


t_all = timed(read_with_payload)
t_one = timed(read_one_column)
print(f"temperature + payload: {t_all:.3f} с")
print(f"только temperature   : {t_one:.3f} с")
print(f"Column pruning дал ускорение в {t_all / t_one:.2f}x")

spark.read.parquet(str(PARQUET_DIR)).select("temperature").agg(F.avg("temperature")).explain()

ANSWER_4_2 = """
TODO: ваш ответ
"""
print(ANSWER_4_2)

## Упражнение 4.3. Проблема мелких файлов

Партиционирование выглядит бесплатным улучшением, но это не так. Запишите набор,
партиционированный по искусственному ключу высокой кардинальности `ts_bucket` (200 значений),
и сравните с партиционированием по `robot_id` (12 значений):

* число созданных файлов;
* суммарный объём;
* средний размер одного файла;
* время чтения всего набора.

In [ ]:
MANY_DIR = ROOT / "parquet_many_partitions"

df_buckets = df_nopay.withColumn(
    "ts_bucket", ((F.col("ts_ms") / 100) % 200).cast("int")
)
# TODO: запишите df_buckets в MANY_DIR с partitionBy("ts_bucket")


def file_stats(path: Path):
    """Возвращает (число файлов .parquet, суммарный размер, средний размер)."""
    # TODO
    raise NotImplementedError


n_few,  s_few,  avg_few  = file_stats(PARTITIONED_DIR)
n_many, s_many, avg_many = file_stats(MANY_DIR)

print(f"{'вариант':<28}{'файлов':>9}{'объём, MiB':>13}{'средний файл, KiB':>20}")
print(f"{'partitionBy(robot_id) [12]':<28}{n_few:>9}{s_few/1024/1024:>13.2f}{avg_few/1024:>20.1f}")
print(f"{'partitionBy(ts_bucket) [200]':<28}{n_many:>9}{s_many/1024/1024:>13.2f}{avg_many/1024:>20.1f}")

t_few  = timed(lambda: spark.read.parquet(str(PARTITIONED_DIR)).count())
t_many = timed(lambda: spark.read.parquet(str(MANY_DIR)).count())
print(f"\nЧтение целиком: 12 партиций {t_few:.3f} с, 200 партиций {t_many:.3f} с")

assert n_many > n_few
assert avg_many < avg_few

## Упражнение 4.4. Технический вывод

Ответьте письменно:

1. Почему слишком большое число мелких partition ухудшает эффективность хранения и чтения?
2. По какому признаку выбирать колонку для `partitionBy`?
3. Когда партиционирование по `robot_id` в нашей задаче окажется бесполезным?

In [ ]:
ANSWER_4_4 = """
TODO: ваш ответ (7-10 предложений)
"""
print(ANSWER_4_4)

---
# Часть 5. TEACH CARD (2 балла)

Заполните паспорт педагогической адаптации выполненной инженерной задачи. Это **обязательная**
часть работы.

**Требования к заполнению.** Формулировки конкретные. «Расскажу про Parquet» не засчитывается.
Нужны измеримый результат, критерий успеха и честно названные ограничения.

**Подсказка по содержанию.** Эксперимент «CSV против Parquet» удобен для школы тем, что результат
виден глазами: два каталога с одними и теми же данными занимают разный объём, и это можно
измерить без единой строки распределённого кода — хватит `du -h`. Аналогия для урока: если
нужна только температура, читается только «стопка температур», а не все листы журнала целиком.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** |  |
| **Целевая аудитория** | 8–9 класс / 10–11 класс / кружок робототехники / СПО |
| **Исследовательский вопрос** |  |
| **Источник данных** |  |
| **Инженерная концепция, сохраняемая без упрощения** |  |
| **Что упрощается относительно университетской лабораторной** |  |
| **Алгоритм действий обучающегося (4–6 шагов)** |  |
| **Измеримый результат / метрика** |  |
| **Критерий успешного выполнения** |  |
| **Вариант усложнения для НТО / хакатона** |  |
| **Риски и ограничения** |  |

---
## Итоговая самопроверка

In [ ]:
checks = {
    "1.1 CSV записан":          CSV_DIR.exists(),
    "1.2 Parquet записан":      len(list(PARQUET_DIR.glob("*.parquet"))) > 0,
    "1.3 partitionBy":          len(part_dirs) == 12,
    "1.4 схема партиции":       "TODO" not in ANSWER_1_4,
    "2.1 размеры и C":          compression_ratio > 1.0,
    "2.2 энтропия":             ratio_nopay > compression_ratio and "TODO" not in ANSWER_2_2,
    "2.3 кодеки":               set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"},
    "2.4 выбор кодека":         "TODO" not in ANSWER_2_4,
    "3.2 бенчмарк чтения":      csv_n == pq_n == part_n,
    "3.3 честное сравнение":    csv_schema_t < csv_t and "TODO" not in ANSWER_3_3,
    "4.1 partition pruning":    "TODO" not in ANSWER_4_1,
    "4.2 column pruning":       t_all > t_one and "TODO" not in ANSWER_4_2,
    "4.3 мелкие файлы":         n_many > n_few,
    "4.4 технический вывод":    "TODO" not in ANSWER_4_4,
}

for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")

print()
print(f"Выполнено: {sum(checks.values())} из {len(checks)}")
print("TEACH CARD проверяется преподавателем вручную.")

In [ ]:
df.unpersist()
spark.stop()
print("SparkSession остановлена.")